# `StructurePreprocessor.get_domains()`

`get_domains` runs a structural domain-segmentation tool and returns a copy of `df_seq` with a per-entry `chopping` string (Merizo / ChainSaw format: domains separated by commas, discontinuous segments joined by underscores, 1-based inclusive residue ranges) plus a `domain_ok` flag. It is the raw, inspectable segmentation that `encode_domains` turns into per-residue channels, mirroring the `get_dssp` to `encode_dssp` pattern.

Two tools are supported: **AFragmenter** (default) builds a residue network from the AlphaFold Predicted Aligned Error (PAE) matrix and finds domains by Leiden clustering, so it needs the PAE sidecars that `fetch_alphafold` downloads; **ChainSaw** predicts boundaries from a PDB / CIF structure and needs a local clone of its repository. Requires `aaanalysis[pro]` (which installs `afragmenter`) and network access for the AlphaFold files.

In [1]:
import warnings
import tempfile
from pathlib import Path
import aaanalysis as aa
aa.options['verbose'] = False
warnings.filterwarnings('ignore')

# Multi-domain human proteins from the bundled gamma-secretase set (UniProt accessions as entries)
df_seq = aa.load_dataset(name='DOM_GSEC', n=10)
df_seq = df_seq[df_seq['entry'].isin(['P05067', 'Q86UE4', 'P01135'])].reset_index(drop=True)

strp = aa.StructurePreprocessor(verbose=False)
af_dir = Path(tempfile.mkdtemp()) / 'alphafold'
strp.fetch_alphafold(df_seq=df_seq, out_folder=af_dir)       # models + PAE sidecars

df_dom = strp.get_domains(df_seq=df_seq, pae_folder=af_dir, tool='afragmenter')
aa.display_df(df_dom[['entry', 'gene', 'chopping', 'domain_ok']], n_rows=10, show_shape=True, char_limit=60)

DataFrame shape: (3, 4)


,entry,gene,chopping,domain_ok
1,Q86UE4,MTDH,"1-44,45-79,80-118,119-165,166-...20-457,458-495,496-537,538-582",True
2,P01135,TGFA,"1-27,28-49,50-86,87-98,99-127,128-144,145-160",True
3,P05067,APP,"1-27,28-123,124-191,192-281,28...63-581,582-698,699-741,742-770",True


The `chopping` column feeds `encode_domains` directly, no folder of chopping files needed:

In [2]:
dict_dom = strp.encode_domains(df_seq=df_dom, features=['domain_boundary', 'n_domains_in_protein'])
print({entry: arr.shape for entry, arr in dict_dom.items()})

{'Q86UE4': (582, 2), 'P01135': (160, 2), 'P05067': (770, 2)}


## Further parameters

`resolution` sets the Leiden community-detection resolution (higher gives more, smaller domains) and `threshold` the PAE cutoff in Angstrom below which two residues are connected in AFragmenter's graph; `on_failure` decides what happens to entries the tool cannot segment (`'nan'` keeps them with an empty `chopping`, `'drop'` removes them, `'raise'` raises). `pdb_folder` and `chainsaw_path` are only read for `tool='chainsaw'`.

In [3]:
df_dom_coarse = strp.get_domains(df_seq=df_seq, pae_folder=af_dir, pdb_folder=af_dir, tool='afragmenter',
                                 chainsaw_path=None, resolution=0.3, threshold=5.0, on_failure='raise')
aa.display_df(df_dom_coarse[['entry', 'chopping']], n_rows=10, show_shape=True, char_limit=60)
# ChainSaw instead (needs the structure files and a local ChainSaw clone):
# df_dom_cs = strp.get_domains(df_seq=df_seq, pdb_folder=af_dir, tool='chainsaw',
#                              chainsaw_path='path/to/chainsaw', on_failure='nan')

DataFrame shape: (3, 2)


,entry,chopping
1,Q86UE4,"1-43,44-84,85-162,163-214,215-...48-402,403-458,459-537,538-582"
2,P01135,"1-30,31-53,54-95,96-144,145-160"
3,P05067,"1-27,28-192,193-281,282-371,372-583,584-696,697-770"
